# 04 - Baseline ML（驗證與視覺化）

批次訓練由 `src/run_baseline_ml.py` 執行：
```bash
cd src && python run_baseline_ml.py
```

訓練內容：
- **特徵**：MFCC 統計量摘要 (94, 39) → 234D (mean/std/max/min/skew/kurtosis)
- **模型**：SVM (RBF) / Random Forest / XGBoost
- **評估**：StratifiedGroupKFold (K=5, group=speaker_id)，防止 speaker leakage

本 notebook 用於**驗證結果與視覺化**。

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results" / "baseline_ml"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

# 載入結果
with open(RESULTS_DIR / "baseline_results.json", encoding="utf-8") as f:
    results = json.load(f)

print(f"Models: {list(results.keys())}")
for name, res in results.items():
    s = res["summary"]
    print(f"  {name}: acc={s['accuracy_mean']:.4f}+/-{s['accuracy_std']:.4f}, "
          f"F1(w)={s['f1_weighted_mean']:.4f}+/-{s['f1_weighted_std']:.4f}")

## 1. 模型比較表

In [ ]:
summary_df = pd.read_csv(RESULTS_DIR / "baseline_summary.csv")
summary_df

## 2. 各 Fold 指標分布

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

model_names = list(results.keys())
metrics = ["accuracy", "f1_weighted", "f1_macro"]
metric_labels = ["Accuracy", "F1 (weighted)", "F1 (macro)"]

fig = make_subplots(rows=1, cols=3, subplot_titles=metric_labels)

for col_idx, (metric, label) in enumerate(zip(metrics, metric_labels), 1):
    for model_name in model_names:
        values = [f[metric] for f in results[model_name]["folds"]]
        fig.add_trace(
            go.Box(y=values, name=model_name, showlegend=(col_idx == 1)),
            row=1, col=col_idx,
        )

fig.update_layout(
    title_text="Baseline ML: 5-Fold Cross-Validation Metrics",
    height=400, width=1000, template="plotly_white",
)
fig.write_html(FIGURES_DIR / "07_baseline_fold_metrics.html")
fig.write_image(FIGURES_DIR / "07_baseline_fold_metrics.png", width=1000, height=400, scale=2)
fig.show()

## 3. 混淆矩陣（以最佳模型的所有 Fold 加總）

In [ ]:
# 找最佳模型（以 F1 macro 平均為準）
best_model = max(results.keys(), key=lambda m: results[m]["summary"]["f1_macro_mean"])
print(f"Best model (by F1 macro): {best_model}")

# 加總所有 fold 的混淆矩陣
cms = results[best_model]["confusion_matrices"]
cm_total = np.sum(cms, axis=0)
classes = results[best_model]["classes"]

# 正規化為百分比
cm_pct = cm_total / cm_total.sum(axis=1, keepdims=True) * 100

# 產生標註文字
text_labels = []
for i in range(len(classes)):
    row = []
    for j in range(len(classes)):
        row.append(f"{cm_total[i][j]}<br>({cm_pct[i][j]:.1f}%)")
    text_labels.append(row)

fig = go.Figure(data=go.Heatmap(
    z=cm_pct,
    x=classes, y=classes,
    text=text_labels, texttemplate="%{text}",
    colorscale="Blues", showscale=True,
    colorbar_title="%",
))
fig.update_layout(
    title=f"{best_model} Confusion Matrix (all folds, row-normalized %)",
    xaxis_title="Predicted", yaxis_title="Actual",
    width=650, height=550, template="plotly_white",
)
fig.write_html(FIGURES_DIR / "08_baseline_confusion_matrix.html")
fig.write_image(FIGURES_DIR / "08_baseline_confusion_matrix.png", width=650, height=550, scale=2)
fig.show()

## 4. Per-class F1 比較

In [ ]:
# 從最後一個 fold 的 classification report 取 per-class F1（近似）
# 更精確：對所有 fold 的 per-class f1 取平均
fig = go.Figure()

for model_name in model_names:
    reports = results[model_name]["classification_reports"]
    per_class_f1 = {c: [] for c in classes}
    for report in reports:
        for c in classes:
            per_class_f1[c].append(report[c]["f1-score"])
    avg_f1 = [np.mean(per_class_f1[c]) for c in classes]
    fig.add_trace(go.Bar(name=model_name, x=classes, y=avg_f1))

fig.update_layout(
    title="Per-class F1 Score (averaged over 5 folds)",
    xaxis_title="Emotion", yaxis_title="F1 Score",
    barmode="group",
    height=450, width=800, template="plotly_white",
    yaxis_range=[0, 1],
)
fig.write_html(FIGURES_DIR / "09_baseline_per_class_f1.html")
fig.write_image(FIGURES_DIR / "09_baseline_per_class_f1.png", width=800, height=450, scale=2)
fig.show()

## 5. Speaker Leakage 驗證

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv(PROJECT_ROOT / "data" / "metadata.csv")
y = df["emotion"].values
groups = df["speaker_id"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

print("Speaker Leakage Check:")
for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
    train_sp = set(groups[train_idx])
    test_sp = set(groups[test_idx])
    overlap = train_sp & test_sp
    print(f"  Fold {fold_idx+1}: train={len(train_sp)} speakers, test={len(test_sp)} speakers, overlap={len(overlap)}")

print("\n[OK] No speaker leakage" if all(
    len(set(groups[tr]) & set(groups[te])) == 0
    for tr, te in sgkf.split(np.zeros(len(y)), y, groups)
) else "\n[WARNING] Speaker leakage detected!")